# CropCare: Phases 1 to 4 on Colab or Kaggle

This notebook runs the CropCare build plan from data to a phone-ready model:

1. **Data**: download PlantVillage and PlantDoc, add your field photos, build the manifest, check for leakage
2. **Baseline**: train on PlantVillage only and test on all three sets
3. **Close the gap**: one technique at a time, compared by PlantDoc test macro F1
4. **Convert**: TFLite files checked on the phone's CPU path, with size and accuracy

**Before you start on Colab**
- Runtime > Change runtime type > **T4 GPU**.
- Add two Colab secrets (key icon on the left): `KAGGLE_USERNAME` and `KAGGLE_KEY`, from kaggle.com > Settings > API > Create New Token. Turn on notebook access for both.
- Put your own photos in Google Drive as `MyDrive/cropcare/field.zip`, holding `images/` and `labels.csv` (columns `filename,label,labelled_by,date,place`). Optional: `MyDrive/cropcare/backgrounds.zip`, 20 or more photos of soil, hands and plants, for background swap.

**Before you start on Kaggle** (no secrets needed: Kaggle signs the notebook in to its own datasets)
- kaggle.com > Code > New Notebook > File > Import Notebook, and paste the GitHub link of this notebook.
- Session options on the right: Accelerator **GPU T4 x2** (one GPU is used) and Internet **on**. Both need a phone-verified Kaggle account.
- Add Input: the dataset `abdallahalidev/plantvillage-dataset`. Optional: upload `field.zip` and `backgrounds.zip` as a private dataset and add it too.
- Use **Save Version > Save & Run All** so the run keeps going with the browser closed. Results land in that version's Output tab.

**Rules this notebook enforces**
- Field photos are never used for training or tuning. They are tested on the baseline once and on the final model once.
- The unsure threshold is tuned on PlantDoc **val**, never on a test set.
- Trained models and every result file are saved (to Drive on Colab, to the Output tab on Kaggle), so a Colab disconnect does not lose work. Re-running a cell skips work that is already saved.

## Settings

In [ ]:
REPO_URL = "https://github.com/Adityaxsahni/cropcare.git"
DRIVE_DIR = "/content/drive/MyDrive/cropcare"   # Colab: results and models are kept here
KAGGLE_DIR = "/kaggle/working/cropcare"         # Kaggle: the same, shown in the notebook's Output tab
ALLOW_CPU = False        # training on a CPU takes many hours, so the notebook stops early without a GPU
MODEL_VERSION = "1.0.0"  # written into the model metadata the app shows
THRESHOLD_TARGET = 0.9   # accuracy wanted on the photos the app does answer
MAX_ACCURACY_DROP = 0.02 # Phase 4: smallest .tflite that loses at most this much accuracy
MIN_AGREEMENT = 0.95     # Phase 4: ...and gives the same answer as the Keras model on at least this share of photos
CHOSEN_FILE = ""         # Phase 4: empty = choose automatically, or "cropcare_float16.tflite", "cropcare_dynamic.tflite", "cropcare_int8.tflite"
QUICK_TEST = False       # True: 1 epoch per stage, to check the notebook end to end
NOTEBOOK_VERSION = 5     # do not edit: lets the setup cell tell an old copy of this notebook from the current one

## Setup

In [ ]:
import csv, itertools, json, os, shutil, subprocess, sys, zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = not IN_COLAB and bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_COLAB or IN_KAGGLE:
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        REPO = Path("/content/cropcare")
        DATA = Path("/content/data")             # local disk: much faster than Drive for training
        RECORDS = Path(DRIVE_DIR)
    else:
        # Everything in /kaggle/working becomes the notebook's output, so code and data stay outside it.
        REPO = Path("/tmp/cropcare")
        DATA = Path("/tmp/cropcare_data")
        RECORDS = Path(KAGGLE_DIR)
    if not (REPO / "ml").is_dir():
        if not REPO_URL:
            raise SystemExit("Set REPO_URL in the settings cell, or upload the cropcare folder to /content/cropcare.")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    else:
        # A runtime that ran an older copy keeps it; update it so fixes on GitHub reach this run.
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "--quiet"], check=False)
    ML = REPO / "ml"
else:                                            # running on your own computer (used to test this notebook)
    ML = Path.cwd()
    DATA = Path(os.environ.get("CROPCARE_DATA", ML.parent / "data"))
    RECORDS = Path(os.environ.get("CROPCARE_RECORDS", DATA / "records"))

RUNS = RECORDS / "runs"
for d in (DATA, RECORDS, RUNS):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(ML)

EPOCHS = ["--epochs-head", "1", "--epochs-finetune", "1"] if QUICK_TEST else []
N_BOOT = "200" if QUICK_TEST else "2000"


def run(script, *args):
    """Run one of the project's scripts and show its output as it happens."""
    cmd = [sys.executable, "-u", f"scripts/{script}", "--data-dir", str(DATA), *map(str, args)]
    print("$", " ".join(cmd[2:]))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        if " I tensorflow/" in line or " W tensorflow/" in line or line.startswith(("I0000", "W0000")):
            continue  # TensorFlow start-up chatter
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{script} failed (exit {proc.returncode})")


def show_md(path):
    from IPython.display import Markdown, display
    display(Markdown(Path(path).read_text(encoding="utf-8")))


import re
_code = subprocess.run(["git", "-C", str(ML), "log", "-1", "--format=%h %cs %s"], capture_output=True, text=True).stdout.strip()
_latest = re.search(r"NOTEBOOK_VERSION = (\d+)", (ML / "CropCare_Colab.ipynb").read_text(encoding="utf-8"))
if _latest and int(_latest.group(1)) > NOTEBOOK_VERSION:
    raise SystemExit(
        f"This copy of the notebook (version {NOTEBOOK_VERSION}) is older than the one on GitHub (version {_latest.group(1)}). "
        "Open it again from GitHub: "
        + ("File > Import Notebook > https://github.com/Adityaxsahni/cropcare/blob/main/ml/CropCare_Colab.ipynb"
           if IN_KAGGLE else
           "https://colab.research.google.com/github/Adityaxsahni/cropcare/blob/main/ml/CropCare_Colab.ipynb"))
print("Platform: ", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local")
print("Code:     ", _code or "unknown", f"| notebook version {NOTEBOOK_VERSION}")
print("ML folder:", ML)
print("Data:     ", DATA)
print("Records:  ", RECORDS)

In [ ]:
if IN_COLAB or IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ai-edge-litert", "pillow"], check=True)
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none")
major, minor = map(int, tf.__version__.split(".")[:2])
if not (2, 16) <= (major, minor) < (2, 21):
    print("Note: this project was tested with TensorFlow 2.16 to 2.20. Newer versions usually work; check the outputs.")
if (IN_COLAB or IN_KAGGLE) and not tf.config.list_physical_devices("GPU") and not (ALLOW_CPU or QUICK_TEST):
    raise SystemExit("No GPU, and training on a CPU takes many hours. " + (
        "Session options > Accelerator > GPU T4 x2, then run the notebook again." if IN_KAGGLE else
        "Runtime > Change runtime type > T4 GPU, then run the notebook again. If Colab says the free GPU "
        "allowance has run out, wait for it to come back or run this notebook on Kaggle (see the top cell)."))

## Phase 1: Data

### 1.1 PlantVillage
Downloads the Kaggle copy (about 4.4 GB, a few minutes; on Kaggle the attached input is used) and copies only the tomato, potato and maize folders, colour and segmented, into the data folder.

In [ ]:
PV = DATA / "raw/plantvillage"
PV_SEG = DATA / "raw/plantvillage_segmented"
CROPS = ("Tomato", "Potato", "Corn")
PV_DONE = DATA / "raw/plantvillage.done"   # written only after a complete copy


def find_kind(src, kind):
    """The `kind` folder holding the crop folders. Looks one and two levels down before walking the
    whole tree, because walking a mounted Kaggle dataset takes minutes."""
    near = [src / kind, *(p / kind for p in sorted(src.iterdir()) if p.is_dir())]
    for p in itertools.chain(near, src.rglob(kind)):
        if p.is_dir() and any(c.name.startswith(CROPS) for c in p.iterdir()):
            return p
    raise SystemExit(f"Could not find the '{kind}' folder in {src}")


if PV_DONE.is_file() or (not IN_KAGGLE and PV.is_dir() and any(PV.iterdir())):
    print("PlantVillage already in place:", PV)
else:
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        except Exception as e:
            raise SystemExit(
                f"Kaggle secrets not available ({type(e).__name__}). Click the key icon on the left, add "
                "KAGGLE_USERNAME and KAGGLE_KEY from your kaggle.json, turn on 'Notebook access' for both, "
                "then run this cell again."
            ) from None
    # On Kaggle an attached copy sits under /kaggle/input (one or more folders down, depending on the layout).
    attached = [p for pattern in ("*plantvillage*", "*/*/*plantvillage*") for p in sorted(Path("/kaggle/input").glob(pattern))
                if p.is_dir()] if IN_KAGGLE else []
    if attached:
        src = attached[0]
    else:
        import kagglehub
        try:
            src = Path(kagglehub.dataset_download("abdallahalidev/plantvillage-dataset"))
        except Exception as e:
            if not IN_KAGGLE:
                raise
            raise SystemExit(f"PlantVillage is not attached ({type(e).__name__}: {e}). "
                             "Add Input > abdallahalidev/plantvillage-dataset, then run the notebook again.") from None
    print("PlantVillage source:", src)
    jobs = []
    for kind, dest in (("color", PV), ("segmented", PV_SEG)):
        for cls in sorted(find_kind(src, kind).iterdir()):
            if cls.is_dir() and cls.name.startswith(CROPS):
                (dest / cls.name).mkdir(parents=True, exist_ok=True)
                jobs += [(f, dest / cls.name / f.name) for f in cls.iterdir()]
    # Many copies at once: a Kaggle dataset is a network mount that copies about 6 files a second one by one.
    print(f"Copying {len(jobs)} images...")
    with ThreadPoolExecutor(64) as pool:
        list(pool.map(lambda job: shutil.copyfile(*job), jobs))
    PV_DONE.write_text(str(len(jobs)), encoding="utf-8")
for d in (PV, PV_SEG):
    if d.is_dir():
        print(d.name, {c.name: sum(1 for _ in c.iterdir()) for c in sorted(d.iterdir())})

### 1.2 PlantDoc

In [ ]:
PD = DATA / "raw/plantdoc"
if not (PD / "test").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pratikkayal/PlantDoc-Dataset", str(PD)], check=True)
print("PlantDoc train folders:", len(list((PD / "train").iterdir())), "| test folders:", len(list((PD / "test").iterdir())))

### 1.3 Your field photos and background photos
Field photos come from `field.zip` in your Drive folder (on Kaggle: in an attached dataset). Without them the notebook still runs, but there is no honest final number.

In [ ]:
def find_upload(name):
    """RECORDS/<name>, or on Kaggle the first attached dataset holding it (up to three folders down)."""
    if IN_KAGGLE:
        for pattern in (f"*/{name}", f"*/*/{name}", f"*/*/*/{name}"):
            hits = sorted(Path("/kaggle/input").glob(pattern))
            if hits:
                return hits[0]
    return RECORDS / name


def unzip_from_drive(name, dest, marker=None):
    """Extract MyDrive/cropcare/<name> into dest once. Accepts a zip whose files sit inside one top folder."""
    z = find_upload(name)
    if not (dest.is_dir() and any(dest.iterdir())):
        if not z.is_file():
            return False
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        entries = list(dest.iterdir())
        if len(entries) == 1 and entries[0].is_dir() and (marker is None or not (dest / marker).exists()):
            top = entries[0]
            for item in list(top.iterdir()):
                shutil.move(str(item), str(dest / item.name))
            top.rmdir()
    return marker is None or (dest / marker).exists()

HAVE_FIELD = unzip_from_drive("field.zip", DATA / "field", marker="labels.csv")
HAVE_BACKGROUNDS = unzip_from_drive("backgrounds.zip", DATA / "backgrounds")
WHERE = "an attached Kaggle dataset" if IN_KAGGLE else "Drive"
print("Field photos:", "yes" if HAVE_FIELD else f"NO (add field.zip to {WHERE} for the final test)")
print("Backgrounds: ", "yes" if HAVE_BACKGROUNDS else "no (background swap will be skipped)")

### 1.4 Manifest, unreadable files and data card

In [ ]:
try:
    run, DATA
except NameError:
    raise SystemExit("The setup cells have not run in this session. Use Runtime > Run all, or run the cells from the top.") from None
problems = []
if not (DATA / "raw/plantvillage").is_dir() or not any((DATA / "raw/plantvillage").iterdir()):
    problems.append("PlantVillage is missing: run step 1.1" + ("." if IN_KAGGLE else " (it needs the KAGGLE_USERNAME and KAGGLE_KEY secrets)."))
if not (DATA / "raw/plantdoc/test").is_dir():
    problems.append("PlantDoc is missing: run step 1.2.")
if problems:
    raise SystemExit(f"Step 1.4 cannot start yet. {DATA} is emptied when the runtime restarts.\n- " + "\n- ".join(problems))

run("prepare_data.py")
run("verify_images.py", "--drop")

### 1.5 Leakage check
Finds val/test photos that are near-duplicates of training photos. Look at the pairs below before dropping them.

In [ ]:
run("check_leakage.py")
import matplotlib.pyplot as plt
from PIL import Image
with open(DATA / "leakage_pairs.csv", newline="", encoding="utf-8") as f:
    pairs = list(csv.DictReader(f))
print(len(pairs), "pairs")
show = pairs[:6]
if show:
    fig, axes = plt.subplots(len(show), 2, figsize=(6, 3 * len(show)), squeeze=False)
    for row, p in zip(axes, show):
        for ax, key in zip(row, ("eval_path", "train_path")):
            ax.imshow(Image.open(DATA / p[key]).convert("RGB")); ax.axis("off")
            ax.set_title(f"{key.split('_')[0]} ({p['distance']} bits)", fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
DROP_LEAKS = True   # set False if the pairs above are different photos
if DROP_LEAKS:
    run("check_leakage.py", "--drop-from-pairs")

In [ ]:
run("data_card.py", "--out", ML / "data_card.md")
for name in ("manifest.csv", "leakage_pairs.csv"):
    shutil.copy(DATA / name, RECORDS / name)
shutil.copy(ML / "data_card.md", RECORDS / "data_card.md")
show_md(ML / "data_card.md")

## Phase 2: Baseline (PlantVillage only)
Training takes a while on a T4. If Colab disconnects, run the notebook again: finished runs are loaded from Drive.

In [ ]:
def train(name, *args):
    out = RUNS / name
    if (out / "model.keras").is_file():
        print(f"{name}: already trained, using {out / 'model.keras'}")
    else:
        run("train.py", *args, *EPOCHS, "--out", out)
    return out / "model.keras"


def evaluate(model, source, split="test"):
    run("evaluate.py", "--model", model, "--source", source, "--split", split, "--n-boot", N_BOOT)
    suffix = "" if Path(model).suffix == ".keras" else f"_{Path(model).stem}"
    return json.loads((Path(model).parent / f"eval_{source}_{split}{suffix}.json").read_text(encoding="utf-8"))


BASELINE = train("p2_baseline", "--sources", "plantvillage")
baseline = {s: evaluate(BASELINE, s) for s in ("plantvillage", "plantdoc")}
if HAVE_FIELD:
    baseline["field"] = evaluate(BASELINE, "field")   # the plan's "before" number on field photos

## Phase 3: Close the lab-to-field gap
Each run adds one change. Background swap runs only if `backgrounds.zip` was found.

In [ ]:
if HAVE_BACKGROUNDS and PV_SEG.is_dir():
    with open(DATA / "manifest.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if not any(r["source"] == "plantvillage_bgswap" for r in rows):
        run("bg_swap.py", "--segmented", PV_SEG, "--backgrounds", DATA / "backgrounds",
            *(["--per-class", "5"] if QUICK_TEST else []))
        run("data_card.py", "--out", ML / "data_card.md")
        shutil.copy(DATA / "manifest.csv", RECORDS / "manifest.csv")
    HAVE_BGSWAP = True
else:
    HAVE_BGSWAP = False
    print("Skipping background swap (no backgrounds or no segmented folder).")

MIX = ["--sources", "plantvillage", "plantdoc", "--weights", "0.6", "0.4"]
MIX_BG = ["--sources", "plantvillage", "plantdoc", "plantvillage_bgswap", "--weights", "0.45", "0.35", "0.2"]
EXPERIMENTS = {"p2_baseline": None, "p3_1_mixed": MIX, "p3_2_strong": [*MIX, "--augment", "strong"]}
if HAVE_BGSWAP:
    EXPERIMENTS["p3_3_bgswap"] = [*MIX_BG, "--augment", "strong"]
    EXPERIMENTS["p3_4_weights"] = [*MIX_BG, "--augment", "strong", "--class-weights"]
else:
    EXPERIMENTS["p3_4_weights"] = [*MIX, "--augment", "strong", "--class-weights"]

phase3 = {"p2_baseline": baseline["plantdoc"]}
for name, args in EXPERIMENTS.items():
    if args is None:
        continue
    phase3[name] = evaluate(train(name, *args), "plantdoc")

In [ ]:
import pandas as pd
table = pd.DataFrame([
    {"run": name, "macro F1": f"{r['macro_f1']:.3f}", "F1 95% CI": f"{r['macro_f1_ci95'][0]:.3f} to {r['macro_f1_ci95'][1]:.3f}",
     "accuracy": f"{r['accuracy']:.1%}", "images": r["n_images"]}
    for name, r in phase3.items()
])
display(table)
BEST_RUN = max(phase3, key=lambda n: phase3[n]["macro_f1"])
print("Best by PlantDoc test macro F1:", BEST_RUN)
print("Overlapping confidence intervals mean the difference may be chance; prefer the simpler run if so.")

### Unsure threshold (tuned on PlantDoc val)
Change `BEST_RUN` here if you prefer a different run from the table above.

In [ ]:
BEST = RUNS / BEST_RUN / "model.keras"
run("tune_threshold.py", "--model", BEST, "--target", 0.0 if QUICK_TEST else THRESHOLD_TARGET,
    *(["--grid-start", "0.05"] if QUICK_TEST else []))
threshold = json.loads((BEST.parent / "threshold.json").read_text(encoding="utf-8"))
print("threshold:", threshold["threshold"])

### Final test on field photos (once)
This cell records which model was tested. It refuses to test a different model afterwards, so the field number stays honest. Delete `final_field_test.json` in Drive only if you really start over.

In [ ]:
marker = RECORDS / "final_field_test.json"
if not HAVE_FIELD:
    print(f"No field photos: add field.zip to {WHERE}, then run from section 1.3 again.")
else:
    if marker.is_file():
        done = json.loads(marker.read_text(encoding="utf-8"))
        if done["model"] != str(BEST):
            raise SystemExit(f"Field photos were already used for the final test of {done['model']} on {done['when']}.")
        print("Final field test already done for this model; showing it again.")
    final_field = evaluate(BEST, "field")
    if not marker.is_file():
        marker.write_text(json.dumps({"model": str(BEST), "when": datetime.now(timezone.utc).isoformat()}, indent=2))
    show_md(BEST.parent / "eval_field_test.md")

## Phase 4: Convert to TFLite
Makes three phone files (dynamic range, float16, full int8) and checks each with LiteRT's XNNPACK delegate, the phone's CPU path. Accuracy is measured on field photos, or on PlantDoc test if you have none yet. Because that set is small, each file is also compared with the Keras model on 600 PlantVillage and PlantDoc test photos ("agreement"). Full int8 is known to break this network (see `ml/scripts/convert.py`); float16 is expected to be chosen.

In [ ]:
export = BEST.parent / "export"
def conversion_format():
    try:
        return json.loads((export / "conversion.json").read_text(encoding="utf-8")).get("format", 1)
    except FileNotFoundError:
        return 0

if conversion_format() < 2:  # missing, or made before float16 and agreement were added
    run("convert.py", "--model", BEST, "--version", MODEL_VERSION, "--eval-source", "field" if HAVE_FIELD else "plantdoc",
        *(["--representative", "20", "--agreement-images", "60"] if QUICK_TEST else []))
show_md(export / "conversion.md")

conv = json.loads((export / "conversion.json").read_text(encoding="utf-8"))
keras, candidates, n = conv["rows"][0], conv["rows"][1:], conv["n_images"]
# On a small test set one photo can be worth more than 2 points, so allow at least one photo.
allowed = max(MAX_ACCURACY_DROP, 1 / n + 1e-9)
print(f"Keras model: accuracy {keras['accuracy']:.1%} on {n} {conv['eval_source']} photos")
print(f"Allowed loss: {allowed * 100:.1f} points (the larger of {MAX_ACCURACY_DROP * 100:.0f} points and one photo), "
      f"and at least {MIN_AGREEMENT:.0%} agreement on {conv['agreement_images']} photos")
for r in candidates:
    lost = (keras["accuracy"] - r["accuracy"]) * 100
    print(f"  {r['file']}: {r['size_bytes'] / 1e6:.2f} MB, accuracy {r['accuracy']:.1%} "
          f"({'lost' if lost >= 0 else 'gained'} {abs(lost):.1f} points), "
          f"agreement with Keras {r['agreement_with_keras']:.1%}")

chosen = None
if CHOSEN_FILE:
    chosen = next((r for r in candidates if r["file"] == CHOSEN_FILE), None)
    if chosen is None:
        raise SystemExit(f"CHOSEN_FILE {CHOSEN_FILE!r} is not one of {[r['file'] for r in candidates]}")
    print("Using CHOSEN_FILE from the settings cell.")
else:
    ok = [r for r in candidates
          if keras["accuracy"] - r["accuracy"] <= allowed and r["agreement_with_keras"] >= MIN_AGREEMENT]
    if ok:
        chosen = min(ok, key=lambda r: r["size_bytes"])

app_model = RECORDS / "app_model"
if chosen is None:
    shutil.rmtree(app_model, ignore_errors=True)  # never leave an older run's model for the app
    print()
    print(f"No .tflite file is within {allowed * 100:.1f} points of the Keras model with at least "
          f"{MIN_AGREEMENT:.0%} agreement, so none was chosen.")
    print("Copy this output into the chat to get it looked at. To override, set CHOSEN_FILE in the settings cell.")
else:
    variant = chosen["file"].removeprefix("cropcare_").removesuffix(".tflite")
    app_model.mkdir(exist_ok=True)
    shutil.copy(export / chosen["file"], app_model / "cropcare.tflite")
    shutil.copy(export / f"cropcare_{variant}.meta.json", app_model / "model_meta.json")
    print(f"Chosen: {chosen['file']} ({chosen['size_bytes'] / 1e6:.2f} MB, accuracy {chosen['accuracy']:.1%})")
    print(f"Copy both files from {app_model} into app/assets/model/ in the repository.")

## Numbers for results.md
Paste these into `results.md`. Phone speed comes later, from the app's About screen.

In [ ]:
def acc(r):
    return f"{r['accuracy']:.1%} ({r['accuracy_ci95'][0]:.1%} to {r['accuracy_ci95'][1]:.1%})"

def f1(r):
    return f"{r['macro_f1']:.3f} ({r['macro_f1_ci95'][0]:.3f} to {r['macro_f1_ci95'][1]:.3f})"

lines = ["## Phase 2: baseline", "", "| Test set | Accuracy (95% CI) | Macro F1 (95% CI) |", "|---|---|---|"]
for s, label in (("plantvillage", "PlantVillage test"), ("plantdoc", "PlantDoc test"), ("field", "Field photos")):
    if s in baseline:
        r = baseline[s]
        lines.append(f"| {label} ({r['n_images']} images) | {acc(r)} | {f1(r)} |")
lines += ["", "## Phase 3", "", "| Run | PlantDoc test macro F1 (95% CI) | PlantDoc test accuracy |", "|---|---|---|"]
lines += [f"| {n} | {f1(r)} | {r['accuracy']:.1%} |" for n, r in phase3.items()]
threshold, final_field = globals().get("threshold"), globals().get("final_field")
conv, chosen = globals().get("conv"), globals().get("chosen")
if threshold:
    t = threshold["threshold"]
    row = next(x for x in threshold["table"] if abs(x["threshold"] - t) < 1e-9)
    lines += ["", f"Unsure threshold: {t:.2f} (PlantDoc val, {threshold['n_val']} images). "
              f"Coverage {row['coverage']:.1%}, accuracy on answered photos {row['accuracy_answered']:.1%}."]
else:
    lines += ["", "Unsure threshold: not tuned yet (run the threshold cell)."]
if final_field:
    lines += [f"Final model on field photos: accuracy {acc(final_field)}, macro F1 {f1(final_field)}."]
else:
    lines += ["Final model on field photos: not run yet."]
lines += ["", "## Phase 4"]
if conv:
    lines += ["", f"Accuracy on {conv['n_images']} {conv['eval_source']} test photos; agreement with the Keras model "
              f"on {conv.get('agreement_images', conv['n_images'])} PlantVillage and PlantDoc test photos.",
              "", "| Version | Size (MB) | Accuracy | Agreement with Keras |", "|---|---|---|---|"]
    lines += [f"| {r['name']} | {r['size_bytes'] / 1e6:.2f} | {r['accuracy']:.1%} | "
              f"{'n/a' if r['agreement_with_keras'] is None else format(r['agreement_with_keras'], '.1%')} |" for r in conv["rows"]]
lines += ["", f"Chosen file: {chosen['file']}" if chosen else "Chosen file: none yet (see the Phase 4 cell output)."]
text = "\n".join(lines)
(RECORDS / "results_snippet.md").write_text(text + "\n", encoding="utf-8")
print(text)

## Download everything in one file
Downloads `cropcare_results.zip` (also kept in your Drive folder; on Kaggle, download it from the Output tab). On your computer, run
`python scripts/import_results.py path/to/cropcare_results.zip` in the cropcare folder: it installs
the model in the app, copies the data card and fills in `results.md`.

In [ ]:
bundle = RECORDS / "cropcare_results.zip"
chosen = globals().get("chosen")
info = {
    "best_run": globals().get("BEST_RUN"),
    "chosen_file": chosen["file"] if chosen else None,
    "model_version": MODEL_VERSION,
    "created": datetime.now(timezone.utc).isoformat(),
}
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("info.json", json.dumps(info, indent=2))
    for name in ("data_card.md", "manifest.csv", "final_field_test.json", "results_snippet.md"):
        if (RECORDS / name).is_file():
            z.write(RECORDS / name, name)
    for f in sorted([*RUNS.glob("*/*.json"), *RUNS.glob("*/export/*.json")]):
        z.write(f, f.relative_to(RECORDS).as_posix())
    for name in ("cropcare.tflite", "model_meta.json"):
        if (RECORDS / "app_model" / name).is_file():
            z.write(RECORDS / "app_model" / name, f"app_model/{name}")
print(f"{bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")
if not chosen:
    print("Note: no model was chosen in Phase 4, so the zip has results but no model for the app.")
if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))
elif IN_KAGGLE:
    print("Download it from this notebook's Output tab (cropcare/cropcare_results.zip).")